# Notebook 4: XGBoost + SHAP Feature Importance

This notebook trains XGBoost models to predict landfill waste and diversion rates,
then uses SHAP (SHapley Additive exPlanations) to identify the most important drivers.

**Outputs:**
- `outputs/ml/charts/shap_beeswarm.png`
- `outputs/ml/charts/shap_bar.png`
- `outputs/ml/shap_results.json`
- `outputs/ml/_state_df_enriched.parquet`

---
## 1. Setup & Data Loading

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

os.makedirs('outputs/ml/charts', exist_ok=True)

SECTOR_COLORS = {
    'Farm': '#7BA17D',
    'Foodservice': '#D4836D',
    'Manufacturing': '#5B8BA0',
    'Residential': '#D4C97A',
    'Retail': '#6BADA0'
}

In [3]:
# Load state summary and policy data
state_df = pd.read_parquet('outputs/analysis/cleaned_data/state_summary.parquet')
policy = pd.read_csv('data/policy_timeline.csv')

state_df = state_df.merge(policy[['state', 'ban_year']], on='state', how='left')
state_df['ban_year'] = state_df['ban_year'].fillna(9999).astype(int)
state_df['ban_active'] = (state_df['year'] >= state_df['ban_year']).astype(int)

# Encode categoricals
le_sector = LabelEncoder()
le_food = LabelEncoder()
le_state = LabelEncoder()
state_df['sector_enc'] = le_sector.fit_transform(state_df['sector'])
state_df['food_type_enc'] = le_food.fit_transform(state_df['food_type'])
state_df['state_enc'] = le_state.fit_transform(state_df['state'])

print(f"State data shape: {state_df.shape}")
state_df.head(3)

State data shape: (216930, 37)


,year,state,sector,sub_sector,sub_sector_category,food_type,tons_surplus,tons_supply,us_dollars_surplus,tons_waste,...,surplus_upstream_100_year_mtch4_footprint,surplus_downstream_100_year_mtch4_footprint,surplus_total_100_year_mtch4_footprint,gallons_water_footprint,meals_wasted,ban_year,ban_active,sector_enc,food_type_enc,state_enc
0,2024,Alabama,Farm,Not Applicable,Not Applicable,Dry Goods,7498.188000,290298.188000,3.551394e+06,7498.188000,...,51.208839,0.000000,51.208839,4.178449e+09,1.249698e+07,9999,0,0,2,0
1,2024,Alabama,Farm,Not Applicable,Not Applicable,Produce,55680.759133,215818.518051,1.577294e+07,53173.425825,...,40.884824,3.474151,44.358975,1.759440e+09,9.208618e+07,9999,0,0,6,0
2,2024,Alabama,Foodservice,Bars And Taverns,Bars And Taverns,Breads & Bakery,3.329335,175.862200,2.048924e+04,3.293845,...,0.016930,0.155164,0.172093,6.418392e+04,5.490296e+03,9999,0,1,0,0


---
## 2. XGBoost for tons_landfill

In [4]:
feature_cols = ['sector_enc', 'food_type_enc', 'state_enc', 'year',
                'tons_surplus', 'tons_supply', 'ban_active']
target = 'tons_landfill'

df_model = state_df[feature_cols + [target]].dropna()
X = df_model[feature_cols]
y = df_model[target]

model = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbosity=0
)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f"Landfill model — CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
model.fit(X, y)

Landfill model — CV R²: 0.9448 ± 0.0568


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

---
## 3. SHAP Analysis & Charts

In [5]:
explainer = shap.TreeExplainer(model)
X_sample = X.sample(min(5000, len(X)), random_state=42)
shap_values = explainer.shap_values(X_sample)

feature_names = []
for col in feature_cols:
    if col.endswith('_enc'):
        feature_names.append(col.replace('_enc', ''))
    else:
        feature_names.append(col)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Feature names: {feature_names}")

SHAP values shape: (5000, 7)
Feature names: ['sector', 'food_type', 'state', 'year', 'tons_surplus', 'tons_supply', 'ban_active']


In [6]:
# Beeswarm plot
try:
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      show=False, max_display=10)
    plt.savefig('outputs/ml/charts/shap_beeswarm.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Saved shap_beeswarm.png")
except Exception as e:
    print(f"Beeswarm failed: {e}")

Saved shap_beeswarm.png


In [7]:
# Bar plot (mean |SHAP|)
try:
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      plot_type='bar', show=False, max_display=10)
    plt.savefig('outputs/ml/charts/shap_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Saved shap_bar.png")
except Exception as e:
    # Manual bar plot fallback
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance = sorted(zip(feature_names, mean_abs_shap), key=lambda x: -x[1])
    fig, ax = plt.subplots(figsize=(10, 6))
    names, vals = zip(*importance)
    ax.barh(range(len(names)), vals[::-1], color='#6BADA0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names[::-1])
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('Feature Importance: Drivers of Landfill Waste (XGBoost + SHAP)')
    plt.savefig('outputs/ml/charts/shap_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Saved shap_bar.png (fallback)")

Saved shap_bar.png


In [8]:
# Compute importance ranking
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_list = sorted(zip(feature_names, mean_abs_shap.tolist()), key=lambda x: -x[1])

print("Feature importance ranking (landfill model):")
for i, (f, v) in enumerate(importance_list):
    print(f"  {i+1}. {f}: {v:.4f}")

Feature importance ranking (landfill model):
  1. tons_surplus: 1924.8870
  2. sector: 481.6585
  3. tons_supply: 400.4296
  4. state: 246.6428
  5. food_type: 221.7620
  6. year: 52.4276
  7. ban_active: 24.7103


---
## 4. Secondary Model (diversion_rate)

In [9]:
state_df['diversion_rate'] = np.where(
    state_df['tons_waste'] > 0,
    1 - (state_df['tons_landfill'] / state_df['tons_waste']),
    0
)

target2 = 'diversion_rate'
df_model2 = state_df[feature_cols + [target2]].dropna()
df_model2 = df_model2[np.isfinite(df_model2[target2])]
X2 = df_model2[feature_cols]
y2 = df_model2[target2]

model2 = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbosity=0
)
cv_scores2 = cross_val_score(model2, X2, y2, cv=5, scoring='r2')
print(f"Diversion model — CV R²: {cv_scores2.mean():.4f} ± {cv_scores2.std():.4f}")
model2.fit(X2, y2)

Diversion model — CV R²: 0.9714 ± 0.0131


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [10]:
explainer2 = shap.TreeExplainer(model2)
X_sample2 = X2.sample(min(5000, len(X2)), random_state=42)
shap_values2 = explainer2.shap_values(X_sample2)
mean_abs_shap2 = np.abs(shap_values2).mean(axis=0)
importance_list2 = sorted(zip(feature_names, mean_abs_shap2.tolist()), key=lambda x: -x[1])

print("Feature importance ranking (diversion model):")
for i, (f, v) in enumerate(importance_list2):
    print(f"  {i+1}. {f}: {v:.6f}")

Feature importance ranking (diversion model):
  1. state: 0.177127
  2. sector: 0.031068
  3. ban_active: 0.028433
  4. tons_surplus: 0.026222
  5. tons_supply: 0.023934
  6. year: 0.012669
  7. food_type: 0.007160


---
## 5. Save Results

In [11]:
# Save shap_results.json
shap_results = {
    "landfill_model": {
        "model": "XGBRegressor",
        "target": "tons_landfill",
        "cv_r2_mean": round(cv_scores.mean(), 4),
        "cv_r2_std": round(cv_scores.std(), 4),
        "n_observations": int(len(X)),
        "feature_importance_shap": [
            {"feature": f, "mean_abs_shap": round(v, 4), "rank": i+1}
            for i, (f, v) in enumerate(importance_list)
        ],
        "key_insight": f"The strongest predictor of landfill waste is {importance_list[0][0]} "
                       f"(mean |SHAP| = {importance_list[0][1]:.1f}), followed by {importance_list[1][0]}."
    },
    "diversion_model": {
        "model": "XGBRegressor",
        "target": "diversion_rate",
        "cv_r2_mean": round(cv_scores2.mean(), 4),
        "cv_r2_std": round(cv_scores2.std(), 4),
        "n_observations": int(len(X2)),
        "feature_importance_shap": [
            {"feature": f, "mean_abs_shap": round(v, 6), "rank": i+1}
            for i, (f, v) in enumerate(importance_list2)
        ],
        "key_insight": f"The strongest predictor of diversion rate is {importance_list2[0][0]} "
                       f"(mean |SHAP| = {importance_list2[0][1]:.4f}), suggesting sector-level "
                       f"infrastructure drives diversion success."
    }
}

with open('outputs/ml/shap_results.json', 'w') as f:
    json.dump(shap_results, f, indent=2)
print("Saved to outputs/ml/shap_results.json")

Saved to outputs/ml/shap_results.json


In [12]:
# Save enriched state_df for downstream use
state_df.to_parquet('outputs/ml/_state_df_enriched.parquet', index=False)
print("Saved to outputs/ml/_state_df_enriched.parquet")

Saved to outputs/ml/_state_df_enriched.parquet
